# Generate Submission

**Purpose:** produce the actual `submission.csv` for the competition, using the best model found
in this project — gender-split logistic regression with 5 features (SeedDiff, WinPctDiff,
AvgMarginDiff, EloDiff, ConfStrengthDiff), the exact setup from notebook 13 (0.1675 average
walk-forward Brier).

**How this differs from every previous notebook:** all previous notebooks held out 2021-2025 to
validate an idea honestly. This one is not a validation test — it's the real production step. The
model gets retrained on *every* historical tournament game through 2025 (using all the data
instead of holding some back), since the actual prediction target, the 2026 tournament, has no
historical precedent to hold out. No new techniques are introduced here (no clipping, no
calibration, no ensembling) — this submission reflects the model exactly as it was validated,
nothing untested added at the last step.

## 1. Load data

Same files as notebook 13, plus the submission template. Regular-season, seed, and conference
files all already include 2026 (68 seeded teams per gender, matching a normal tournament field),
which is what makes it possible to compute this season's features.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")

m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
m_reg = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
w_reg = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
m_conf_teams = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
w_conf_teams = pd.read_csv(DATA_DIR / "WTeamConferences.csv")
sample_sub = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

print(f"historical men's tourney games (training labels): {len(m_tourney):,}")
print(f"historical women's tourney games (training labels): {len(w_tourney):,}")
print(f"men's 2026 seeds: {(m_seeds['Season'] == 2026).sum()}, women's 2026 seeds: {(w_seeds['Season'] == 2026).sum()}")
print(f"submission rows to predict: {len(sample_sub):,}")


historical men's tourney games (training labels): 2,585
historical women's tourney games (training labels): 1,717
men's 2026 seeds: 68, women's 2026 seeds: 68
submission rows to predict: 132,133


## 2. Whole-season team features, Elo, seeds, and conference strength (unchanged from notebook 13)

In [2]:
def team_season_stats(reg):
    won = reg[["Season", "WTeamID", "WScore", "LScore"]].rename(
        columns={"WTeamID": "TeamID", "WScore": "PF", "LScore": "PA"})
    won["Win"] = 1
    lost = reg[["Season", "LTeamID", "LScore", "WScore"]].rename(
        columns={"LTeamID": "TeamID", "LScore": "PF", "WScore": "PA"})
    lost["Win"] = 0
    games = pd.concat([won, lost], ignore_index=True)
    stats = games.groupby(["Season", "TeamID"]).agg(
        GamesPlayed=("Win", "size"), WinPct=("Win", "mean"),
        AvgPF=("PF", "mean"), AvgPA=("PA", "mean"),
    ).reset_index()
    stats["AvgScoreMargin"] = stats["AvgPF"] - stats["AvgPA"]
    return stats

team_stats = pd.concat([team_season_stats(m_reg), team_season_stats(w_reg)], ignore_index=True)

def seed_num(s):
    return int("".join(ch for ch in s if ch.isdigit()))

m_seeds["SeedNum"] = m_seeds["Seed"].apply(seed_num)
w_seeds["SeedNum"] = w_seeds["Seed"].apply(seed_num)
seeds = pd.concat([m_seeds[["Season", "TeamID", "SeedNum"]], w_seeds[["Season", "TeamID", "SeedNum"]]], ignore_index=True)
seed_idx = seeds.set_index(["Season", "TeamID"])["SeedNum"]

def compute_elo(compact_df, k=20, home_adv=100, mean_reversion=0.75):
    elo = {}
    elo_by_season_end = {}
    for season in sorted(compact_df["Season"].unique()):
        for team in elo:
            elo[team] = elo[team] * mean_reversion + 1500 * (1 - mean_reversion)
        season_games = compact_df[compact_df["Season"] == season].sort_values("DayNum")
        for game in season_games.itertuples(index=False):
            w, l = game.WTeamID, game.LTeamID
            if w not in elo: elo[w] = 1500
            if l not in elo: elo[l] = 1500
            w_elo, l_elo = elo[w], elo[l]
            if game.WLoc == "H": w_elo += home_adv
            elif game.WLoc == "A": l_elo += home_adv
            w_exp = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
            mov = game.WScore - game.LScore
            mov_mult = np.log(max(mov, 1) + 1) * (2.2 / ((w_elo - l_elo) * 0.001 + 2.2))
            elo[w] += k * mov_mult * (1 - w_exp)
            elo[l] -= k * mov_mult * (1 - w_exp)
        for team in elo:
            elo_by_season_end[(season, team)] = elo[team]
    return elo_by_season_end

m_elo = compute_elo(m_reg)
w_elo = compute_elo(w_reg)
elo_idx = pd.Series({**m_elo, **w_elo})
elo_idx.index = pd.MultiIndex.from_tuples(elo_idx.index, names=["Season", "TeamID"])

conf_teams = pd.concat([m_conf_teams, w_conf_teams], ignore_index=True)
conf_teams["Elo"] = conf_teams.apply(lambda r: elo_idx.get((r["Season"], r["TeamID"]), np.nan), axis=1)
conf_teams_valid = conf_teams.dropna(subset=["Elo"])
conf_strength = conf_teams_valid.groupby(["Season", "ConfAbbrev"])["Elo"].mean().rename("ConfStrength")
team_conf_strength = conf_teams_valid.merge(conf_strength, on=["Season", "ConfAbbrev"], how="left")
conf_strength_idx = team_conf_strength.set_index(["Season", "TeamID"])["ConfStrength"]

print(f"team-season stats: {len(team_stats):,}, seeds: {len(seeds):,}, Elo entries: {len(elo_idx):,}, conf-strength entries: {len(conf_strength_idx):,}")
print(f"2026 teams with an Elo rating: {elo_idx.loc[2026].shape[0] if 2026 in elo_idx.index.get_level_values('Season') else 0}")


team-season stats: 23,604, seeds: 4,506, Elo entries: 24,158, conf-strength entries: 23,606
2026 teams with an Elo rating: 751


## 3. Build the training matchup table (historical tournament games only)

Same as notebook 13 — one row per historical tournament game (2026 has none yet, since it hasn't
been played), with all 5 features joined and the gender tag kept for training two separate
models.

In [3]:
def build_matchups(tourney, gender):
    df = tourney.copy()
    df["Team1"] = df[["WTeamID", "LTeamID"]].min(axis=1)
    df["Team2"] = df[["WTeamID", "LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["Team1"]).astype(int)
    df["Gender"] = gender
    return df[["Season", "Team1", "Team2", "Label", "Gender"]]

def add_features(matchups):
    matchups = matchups.copy()
    matchups["Seed1"] = matchups.apply(lambda r: seed_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["Seed2"] = matchups.apply(lambda r: seed_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["SeedDiff"] = matchups["Seed2"] - matchups["Seed1"]
    matchups["Elo1"] = matchups.apply(lambda r: elo_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["Elo2"] = matchups.apply(lambda r: elo_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["EloDiff"] = matchups["Elo1"] - matchups["Elo2"]
    matchups["ConfStrength1"] = matchups.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["ConfStrength2"] = matchups.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["ConfStrengthDiff"] = matchups["ConfStrength1"] - matchups["ConfStrength2"]
    for team_col, suffix in [("Team1", "1"), ("Team2", "2")]:
        joined = matchups.merge(team_stats, left_on=["Season", team_col], right_on=["Season", "TeamID"], how="left")
        matchups[f"WinPct{suffix}"] = joined["WinPct"].values
        matchups[f"AvgMargin{suffix}"] = joined["AvgScoreMargin"].values
    matchups["WinPctDiff"] = matchups["WinPct1"] - matchups["WinPct2"]
    matchups["AvgMarginDiff"] = matchups["AvgMargin1"] - matchups["AvgMargin2"]
    return matchups

FEATURES = ["SeedDiff", "WinPctDiff", "AvgMarginDiff", "EloDiff", "ConfStrengthDiff"]

training_matchups = add_features(
    pd.concat([build_matchups(m_tourney, "M"), build_matchups(w_tourney, "W")], ignore_index=True)
)
training_matchups = training_matchups.dropna(subset=FEATURES).reset_index(drop=True)
print(f"training rows: {len(training_matchups):,} (men's: {(training_matchups['Gender']=='M').sum():,}, women's: {(training_matchups['Gender']=='W').sum():,})")


training rows: 4,302 (men's: 2,585, women's: 1,717)


## 4. Train the final models on all historical data

No held-out fold this time — every historical tournament game through 2025 is used, since the
goal now is the best possible model for predicting a season with no precedent to hold back.

In [4]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def fit_logreg(X, y, l2=1.0, n_iter=50, tol=1e-8):
    n, p = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])
    w = np.zeros(p + 1)
    reg = np.eye(p + 1) * l2
    reg[0, 0] = 0.0
    for _ in range(n_iter):
        pr = sigmoid(Xb @ w)
        grad = Xb.T @ (pr - y) + reg @ w
        W = np.clip(pr * (1 - pr), 1e-6, None)
        H = Xb.T @ (Xb * W[:, None]) + reg
        step = np.linalg.solve(H, grad)
        w_new = w - step
        if np.max(np.abs(w_new - w)) < tol:
            w = w_new
            break
        w = w_new
    return w

def predict_proba(X, w):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    return sigmoid(Xb @ w)

men_train = training_matchups[training_matchups["Gender"] == "M"]
women_train = training_matchups[training_matchups["Gender"] == "W"]

w_men_final = fit_logreg(men_train[FEATURES].values, men_train["Label"].values)
w_women_final = fit_logreg(women_train[FEATURES].values, women_train["Label"].values)

print("final men's model coefficients (intercept, then SeedDiff/WinPctDiff/AvgMarginDiff/EloDiff/ConfStrengthDiff):")
print(np.round(w_men_final, 5))
print("final women's model coefficients:")
print(np.round(w_women_final, 5))


final men's model coefficients (intercept, then SeedDiff/WinPctDiff/AvgMarginDiff/EloDiff/ConfStrengthDiff):
[-0.01612  0.0351  -1.21489  0.05297  0.00453  0.00238]
final women's model coefficients:
[ 0.09771  0.09946 -1.69989  0.06306  0.00451  0.00331]


## 5. Compute 2026 features and predict for every submission row

Every row's `ID` is parsed into `Season_Team1_Team2` (`Team1` is always the lower TeamID, matching
the format Kaggle requires and the same convention used throughout this project). The men's model
is used for TeamIDs under 2000, the women's model for TeamIDs 3000 and up. Most of the 132,133
rows involve at least one team that didn't make the 2026 tournament (no seed, since only 68 teams
per gender have one) — those get a neutral 0.5, matching the sample template's own default,
since only pairs between two actual 2026 tournament teams are practically scored.

In [5]:
ids = sample_sub["ID"].str.split("_", expand=True).astype(int)
ids.columns = ["Season", "Team1", "Team2"]
sub = pd.concat([sample_sub[["ID"]], ids], axis=1)

sub["Seed1"] = sub.apply(lambda r: seed_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["Seed2"] = sub.apply(lambda r: seed_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["SeedDiff"] = sub["Seed2"] - sub["Seed1"]
sub["Elo1"] = sub.apply(lambda r: elo_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["Elo2"] = sub.apply(lambda r: elo_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["EloDiff"] = sub["Elo1"] - sub["Elo2"]
sub["ConfStrength1"] = sub.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["ConfStrength2"] = sub.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["ConfStrengthDiff"] = sub["ConfStrength1"] - sub["ConfStrength2"]
for team_col, suffix in [("Team1", "1"), ("Team2", "2")]:
    joined = sub.merge(team_stats, left_on=["Season", team_col], right_on=["Season", "TeamID"], how="left")
    sub[f"WinPct{suffix}"] = joined["WinPct"].values
    sub[f"AvgMargin{suffix}"] = joined["AvgScoreMargin"].values
sub["WinPctDiff"] = sub["WinPct1"] - sub["WinPct2"]
sub["AvgMarginDiff"] = sub["AvgMargin1"] - sub["AvgMargin2"]

has_all_features = sub[FEATURES].notna().all(axis=1)
is_men = sub["Team1"] < 2000

pred = np.full(len(sub), 0.5)
men_mask = has_all_features & is_men
women_mask = has_all_features & ~is_men
pred[men_mask.values] = predict_proba(sub.loc[men_mask, FEATURES].values, w_men_final)
pred[women_mask.values] = predict_proba(sub.loc[women_mask, FEATURES].values, w_women_final)

sub["Pred"] = pred

print(f"rows with a real (non-default) prediction: {has_all_features.sum():,} of {len(sub):,}")
print(f"  men's: {men_mask.sum():,}, women's: {women_mask.sum():,}")
print(f"prediction range: {sub['Pred'].min():.4f} to {sub['Pred'].max():.4f}")


rows with a real (non-default) prediction: 4,556 of 132,133
  men's: 2,278, women's: 2,278
prediction range: 0.0009 to 0.9995


## 6. Sanity checks before saving

A few spot checks: predictions should stay in [0, 1], the real (non-default) row count should
roughly match all possible pairs among the 68 tournament teams per gender
(68 x 67 / 2 = 2,278 pairs each), and a heavy-favorite matchup (a 1-seed against a 16-seed) should
predict close to the historical norm, not something wild.

In [6]:
expected_pairs_per_gender = 68 * 67 // 2
print(f"expected real-prediction rows per gender (68 choose 2): {expected_pairs_per_gender:,}")
print(f"actual: men's {men_mask.sum():,}, women's {women_mask.sum():,}")

assert sub["Pred"].between(0, 1).all(), "found a prediction outside [0, 1]"
print("all predictions within [0, 1]: OK")

# spot check: pull a real 1-seed vs 16-seed matchup from the 2026 men's field and check its prediction
seed1_2026 = m_seeds[(m_seeds["Season"] == 2026) & (m_seeds["SeedNum"] == 1)]
seed16_2026 = m_seeds[(m_seeds["Season"] == 2026) & (m_seeds["SeedNum"] == 16)]
if len(seed1_2026) and len(seed16_2026):
    t1, t16 = seed1_2026.iloc[0]["TeamID"], seed16_2026.iloc[0]["TeamID"]
    lo, hi = min(t1, t16), max(t1, t16)
    row = sub[(sub["Team1"] == lo) & (sub["Team2"] == hi)]
    if len(row):
        favored_is_low = t1 == lo
        p_low = row["Pred"].values[0]
        p_favorite = p_low if favored_is_low else 1 - p_low
        print(f"\nsample 1-seed vs 16-seed check: predicted P(1-seed wins) = {p_favorite:.3f}")


expected real-prediction rows per gender (68 choose 2): 2,278
actual: men's 2,278, women's 2,278
all predictions within [0, 1]: OK

sample 1-seed vs 16-seed check: predicted P(1-seed wins) = 0.973


## 7. Save the submission file

In [7]:
submission_final = sub[["ID", "Pred"]]
submission_final.to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(submission_final):,} rows")
print(submission_final.head(3).to_string(index=False))


wrote submission.csv: 132,133 rows
            ID  Pred
2026_1101_1102   0.5
2026_1101_1103   0.5
2026_1101_1104   0.5


## Conclusion

**submission.csv generated successfully, all sanity checks passed.** 4,556 of the 132,133 rows
got a real, model-based prediction (2,278 men's + 2,278 women's — exactly matching 68 choose 2
per gender, the number of possible pairs among a 68-team field), confirming the seed-based
filtering worked exactly as intended. Every other row defaults to 0.5, matching the sample
template.

All predictions fall within [0, 1] (range 0.0009 to 0.9995), and the 1-seed vs. 16-seed spot
check predicted a 97.3% win probability for the 1-seed — in line with the historical norm for
that matchup, not an extreme or broken number.

This model (gender-split logistic regression, 5 features, trained on all historical tournament
games through 2025) is the same one validated at 0.1675 average Brier across the 2021-2025
walk-forward folds in notebook 13 — nothing new or untested was introduced at this final step.